# Setup

In [2]:
!pip install -q modelscan

In [3]:
!pip install -q xgboost==1.7.6
!pip install -U -q scikit-learn==1.3.2

In [4]:
import pickle
from pathlib import Path
import os
import numpy as np
from pickle_codeinjection import generate_unsafe_file
from xgboost_diabetes_model import train_model, get_predictions

# Save a XGBoost Model

The model is trained on a diabetes dataset, and predicts whether a person has diabetes or not. The dataset can be found here: [Link to PIMA Indian diabetes dataset](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database). The model is saved at ```./XGBoostModels/safe_model.pkl```

In [5]:
model_directory = os.path.join(os.getcwd(), "XGboostModels")
if not os.path.isdir(model_directory):
    os.mkdir(model_directory)

safe_model_path_pickle = os.path.join(model_directory, "safe_model.pkl")
model = train_model()
with open(safe_model_path_pickle, "wb") as fo:
    pickle.dump(model, fo)

# Predict using Safe Model

In [6]:
number_of_predictions = 3
get_predictions(number_of_predictions, model)

The model predicts: [0, 1, 1]
The true labels are: [0. 1. 1.]


# Scan the safe model

The scan results include information on the files scanned, and any issues if found. For the safe model scanned, modelscan finds no code injections in it, as expected.

In [7]:
!modelscan -p XGboostModels\safe_model.pkl

No settings file detected at c:\Users\tugut\Documents\GitHub\First\modelscan-settings.toml. Using defaults. 

Scanning c:\Users\tugut\Documents\GitHub\First\XGboostModels\safe_model.pkl using modelscan.scanners.PickleUnsafeOpScan model scan

--- Summary ---

 No issues found! 🎉


# Model Serialization Attack

Here code is injected in the safe model to read aws secret keys. The unsafe model is saved at ```./XGBoostModels/unsafe_model.pkl```

In [8]:
# Inject code with the command
command = "system"
malicious_code = """cat ~/.aws/secrets
    """

In [9]:
with open(safe_model_path_pickle, "rb") as fo:
    safe_model_pickle = pickle.load(fo)

unsafe_model_path = os.path.join(model_directory, "unsafe_model.pkl")
generate_unsafe_file(model, command, malicious_code, unsafe_model_path)

# Predict using Unsafe Model

The malicious code gets executed when the model is loaded. The aws secret keys are displayed. 

Also, the unsafe model predicts just as well as safe model i.e., the code injection attack will not impact the model performance. The unaffected performance of unsafe models makes the ML models an effective attack vector. 

In [10]:
with open(unsafe_model_path, "rb") as fo:
    unsafe_model = pickle.load(fo)

get_predictions(number_of_predictions, unsafe_model)

The model predicts: [0, 1, 1]
The true labels are: [0. 1. 1.]


# Scan the Unsafe Model

The scan results include information on the files scanned, and any issues if found. In this case, a critical severity level issue is found in the unsafe model scanned. 

modelscan also outlines the found operator(s) and module(s) deemed unsafe. 

In [12]:
!modelscan -p XGboostModels\unsafe_model.pkl

No settings file detected at c:\Users\tugut\Documents\GitHub\First\modelscan-settings.toml. Using defaults. 

Scanning c:\Users\tugut\Documents\GitHub\First\XGboostModels\unsafe_model.pkl using modelscan.scanners.PickleUnsafeOpScan model scan

--- Summary ---

Total Issues: 1

Total Issues By Severity:

    - LOW: 0
    - MEDIUM: 0
    - HIGH: 0
    - CRITICAL: 1

--- Issues by Severity ---

--- CRITICAL ---

Unsafe operator found:
  - Severity: CRITICAL
  - Description: Use of unsafe operator 'system' from module 'nt'
  - Source: c:\Users\tugut\Documents\GitHub\First\XGboostModels\unsafe_model.pkl
